### Langchain

In [1]:
!pip install -qU langchain langchain-huggingface langchain-qdrant langchain-groq langchain-text-splitters
!pip install -q sentence-transformers qdrant-client


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [3]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "ensurepip", "--default-pip"])

CompletedProcess(args=['C:\\Users\\Shraddha\\PycharmProjects\\RAG\\.venv\\Scripts\\python.exe', '-m', 'ensurepip', '--default-pip'], returncode=0)

In [4]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "langchain-groq"])

CompletedProcess(args=['C:\\Users\\Shraddha\\PycharmProjects\\RAG\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'langchain-groq'], returncode=0)

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [6]:
resp = llm.invoke("What is the capital of India?")
print(resp.content)

The capital of India is **New Delhi**.


In [7]:
response = llm.invoke(
    [
      (
          "system",
          "You are a helpful assistant that reviews python code and suggests improvement",
      ),
      ("human", """
  def process_data(data_list):
    for item in data_list:
        if "@" in item:
            print("Email:", item)
        else:
            try:
                num = int(item)
                print("Number:", num * 2)
            except:
                print("Other:", item)

      """),
    ]
)
print("This is Response", response.content)

This is Response **Code review & suggestions**

Below is the original snippet (with the indentation problem highlighted):

```python
def process_data(data_list):
    for item in data_list:
        if "@" in item:
            print("Email:", item)
        else:
            try:
                num = int(item)
                print("Number:", num * 2)
            except:
                print("Other:", item)
```

### 1️⃣ What’s good?
* The logic is clear:  
  * Detect strings that look like e‑mails (`"@"` in the text).  
  * Try to treat everything else as an integer and double it.  
  * Fallback to a generic “Other” case.
* The function is short and easy to read.

---

### 2️⃣ What can be improved?

| Area | Issue | Why it matters | Suggested fix |
|------|-------|----------------|---------------|
| **Indentation / formatting** | The original code had a stray space before the closing line (`}`) which makes the file uncompilable. | Python is indentation‑sensitive; a syntax error stops th

### RAG in Langchain

In [8]:
import requests
from langchain_core.documents import Document

WEB_URL = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt"

response = requests.get(WEB_URL)
response.raise_for_status()

docs = [Document(page_content=response.text, metadata={"source": WEB_URL})]
print(docs)

[Document(metadata={'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt'}, page_content='# AtliqAI HR Policies\n\nAtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.\n\n---\n\n## Employment & Onboarding\n\n### Offer and Joining Formalities\n\nUpon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will share a pre-joining checklist that includes submission of educational certificates, identity proof, address proof, previous employment documents, and a recent photograph. Failure to submit required documents within 7 working days of joining may result in withholding of the

In [9]:
!pip install tiktoken


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=100, chunk_overlap=0
)
documents = text_splitter.split_documents(docs)
len(documents)

for i, doc in enumerate(documents):
    print(f"--- Document C"
          f"hunk {i+1} ---")
    print(f"Metadata: {doc.metadata}")
    print(f"Content:\n{doc.page_content}\n")

--- Document Chunk 1 ---
Metadata: {'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt'}
Content:
# AtliqAI HR Policies

AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.

---

## Employment & Onboarding

### Offer and Joining Formalities

--- Document Chunk 2 ---
Metadata: {'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt'}
Content:
Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will share a pre-joining checklist that includes submission of educational certificates, identity proof, address proof, pr

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

embedder = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13992.33it/s]


In [14]:
vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embedder,
    path="/tmp/my_lang_vs",
    collection_name="hr_docs",
)

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [16]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. Cite section names. Say 'I don't know' if unsure."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [18]:
def format_docs(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        dl_meta  = doc.metadata.get("dl_meta", {})
        parts.append(f"[{i}]: \n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

In [19]:
def rag(query: str) -> str:
    docs         = retriever.invoke(query)
    context      = format_docs(docs)
    prompt_value = RAG_PROMPT.invoke({"context": context, "question": query})
    response     = llm.invoke(prompt_value)
    return response.content

In [21]:
for q in [
        "How many casual leaves am I entitled to?",
        "What is the notice period for Band 4 employees?",
        "How long is the probation period?",
    ]:
        print(f"\nQ: {q}\nA: {rag(q)}\n")


Q: How many casual leaves am I entitled to?
A: You are entitled to **12 casual leaves per calendar year** (credited at 1 leave per month).【1】


Q: What is the notice period for Band 4 employees?
A: The notice period for Band 4 employees is **60 days**【1】.


Q: How long is the probation period?
A: The probation period is **6 months** from the date of joining. (See section [1])



In [22]:
docs_1 = retriever.invoke("How many casual leaves am I entitled to?")

In [23]:
context      = format_docs(docs_1)
print(context)

[1]: 
Every confirmed employee is entitled to 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unplanned absences. A maximum of 3 consecutive casual leaves can be taken at a time. Casual leaves cannot be carried forward to the next calendar year and lapse on December 31st.

### Sick Leave

---

[2]: 
Employees are entitled to 10 sick leaves per calendar year. Sick leave can be availed in case of illness, hospitalisation, or medical procedures. A medical certificate from a registered practitioner is mandatory for sick leave of more than 2 consecutive days. Unused sick leaves up to a maximum of 10 can be carried forward to the following year.

### Earned Leave

---

[3]: 
Employees who are unable to serve the full notice period may request a notice period buy-out, subject to management approval. The buy-out amount is calculated at the basic daily rate for each day of unserved notice. The amount will be 